In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.databricks.execution.timeout", "18000")

BRONZE_TRAIN = "dbr_dev_ua5816bd.gulidov_artem_bronze.bronze_train"
BRONZE_VALIDATION = "dbr_dev_ua5816bd.gulidov_artem_bronze.bronze_validation"

SILVER_TRAIN = "dbr_dev_ua5816bd.gulidov_artem_silver.silver_train"
SILVER_VALIDATION = "dbr_dev_ua5816bd.gulidov_artem_silver.silver_validation"

def clean_squad_table(source_table, target_table):
    df = spark.table(source_table)

    text_columns = ["title", "context", "question"]

    for column in text_columns:
        df = df.withColumn(
            column,
            F.trim(
                F.regexp_replace(
                    F.regexp_replace(F.col(column), r"<[^>]+>", " "), 
                    r"\s+",
                    " "
                )
            )
        )

    for column in text_columns:
        df = df.withColumn(
            column,
            F.when(F.col(column) == "", None).otherwise(F.col(column))
        )

    df = (
        df
        .withColumn(
            "answer_text",
            F.get(F.col("answers.text"), 0)
        )
        .withColumn(
            "answer_start",
            F.get(F.col("answers.answer_start"), 0)
        )
        .drop("answers")
    )

    df = df.withColumn(
        "answer_text",
        F.when(
            F.col("answer_text") == "",
            None
        ).otherwise(F.trim(F.col("answer_text")))
    )

    df = df.withColumn(
        "is_impossible",
        F.col("answer_text").isNull()
    )

    df = df.filter(
        F.col("id").isNotNull() &
        F.col("context").isNotNull() &
        F.col("question").isNotNull() &
        F.col("title").isNotNull()
    )

    df = df.dropDuplicates(["id"])

    df = (
        df
        .withColumn("_processed_at", F.current_timestamp())
        .withColumn("_source_table", F.lit(source_table))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Processed {source_table} - {target_table}")
    print(f"Records: {spark.table(target_table).count():,}")

clean_squad_table(BRONZE_TRAIN, SILVER_TRAIN)
clean_squad_table(BRONZE_VALIDATION, SILVER_VALIDATION)

In [0]:
%sql
SELECT COUNT(*) FROM dbr_dev_ua5816bd.gulidov_artem_silver.silver_validation WHERE is_impossible = FALSE